# Step 1: EDA & Data Cleaning
This notebook covers the initial exploration and cleaning of the Costa Rican Household Poverty dataset. We handle missing values, map mixed-type columns, and prepare the data for feature engineering.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import os

## 1. Loading the Data
We load both the training and test datasets.

In [ ]:
DATA_PATH = '../data/'
train = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_PATH, 'test.csv'))
print(f"Train shape: {train.shape}, Test shape: {test.shape}")

## 2. Cleaning Function
We define a unified cleaning function for both datasets.

In [ ]:
def clean_poverty_data(df):
    """Perform initial cleaning and type conversion."""
    df = df.copy()
    
    # 1. Map mixed-type columns (yes=1, no=0)
    # edjefe/edjefa: years of education of male/female head of household
    # dependency: Dependency rate, calculated = (number of members of the household under 19 or over 64)/(number of member of household between 19 and 64)
    mapping = {'yes': 1, 'no': 0}
    df['edjefe'] = df['edjefe'].replace(mapping).astype(float)
    df['edjefa'] = df['edjefa'].replace(mapping).astype(float)
    df['dependency'] = df['dependency'].replace(mapping).astype(float)
    
    # 2. Impute missing values based on domain logic
    # v2a1: monthly rent payment (if tipovivi1=1, they own the house, so rent is 0)
    df.loc[(df['tipovivi1'] == 1), 'v2a1'] = 0
    
    # v18q1: number of tablets (if v18q=0, they have 0 tablets)
    df.loc[(df['v18q'] == 0), 'v18q1'] = 0
    
    # rez_esc: Years behind in school (fill with 0 if missing)
    df['rez_esc'] = df['rez_esc'].fillna(0)
    
    # meaneduc: average years of education for adults (18+) (fill with 0 if missing)
    df['meaneduc'] = df['meaneduc'].fillna(0)
    df['SQBmeaned'] = df['SQBmeaned'].fillna(0)
    
    return df

## 3. Running the Cleaning
We apply the function and save the processed files.

In [ ]:
train_cleaned = clean_poverty_data(train)
test_cleaned = clean_poverty_data(test)

PROCESSED_PATH = '../data/processed/'
os.makedirs(PROCESSED_PATH, exist_ok=True)

train_cleaned.to_csv(os.path.join(PROCESSED_PATH, 'train_cleaned.csv'), index=False)
test_cleaned.to_csv(os.path.join(PROCESSED_PATH, 'test_cleaned.csv'), index=False)

print("Cleaned files saved to ../data/processed/")